# 🥉 Couche Bronze : Ingestion Massive depuis AWS S3 Open Data

---

## 🎯 Objectifs de ce Notebook
1. Comprendre la philosophie et les règles d'or de la **Couche Bronze** (Raw Landing Zone).
2. Découvrir le jeu de données public **NYC TLC Taxi Data** hébergé sur AWS Open Data.
3. Télécharger et ingérer les données brutes dans le stockage objet local **MinIO** (Bucket `bronze`).
4. Télécharger et stocker la table de référence dimensionnelle des zones géographiques (`taxi_zone_lookup.csv`).
5. Valider l'intégrité et la volumétrie sans altérer les données brutes.


---
## 1. Les Règles d'Or de la Couche Bronze

En ingénierie des données moderne :
- **Immutabilité absolue** : On n'altère **jamais** les données brutes. Pas de filtrage, pas de déduplication, pas de renommage de colonne à ce stade.
- **Auditabilité / Rejouabilité** : Si un bug est découvert 6 mois plus tard dans un calcul métier, on doit pouvoir ré-exécuter le pipeline complet depuis les données brutes exactes.
- **Traçabilité** : On horodate et structure les fichiers bruts par date d'ingestion ou période source (`year=YYYY/month=MM/`).
- **Format natif** : On conserve le format d'origine (ici le format Parquet optimisé fourni par AWS Open Data).


---
## 2. Source de Données : NYC Taxi and Limousine Commission (TLC)

AWS héberge gratuitement le registre public des courses de taxis de New York :
- **Fichier des trajets (Yellow Taxi)** : ~3 millions de courses par mois contenant l'heure exacte de prise en charge/dépose, coordonnées des zones, distances, montants détaillés (pourboires, taxes, suppléments).
- **Fichier de référence (Zones)** : Table CSV associant chaque `LocationID` (1 à 265) à son `Borough` (Manhattan, Brooklyn, Queens, Bronx, Staten Island) et son quartier.


In [ ]:
import os
import requests
from tqdm import tqdm
import boto3

# Configuration des sources AWS Open Data
DATASET_YEAR = "2024"
DATASET_MONTH = "01"

TAXI_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{DATASET_YEAR}-{DATASET_MONTH}.parquet"
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Chemins de destination locaux temporaires
RAW_DIR = "/workspace/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

taxi_local_path = os.path.join(RAW_DIR, f"yellow_tripdata_{DATASET_YEAR}-{DATASET_MONTH}.parquet")
zone_local_path = os.path.join(RAW_DIR, "taxi_zone_lookup.csv")

print(f"🔗 URL Taxis : {TAXI_URL}")
print(f"🔗 URL Zones : {ZONE_URL}")


---
## 3. Téléchargement des Données avec Barre de Progression


In [ ]:
def download_file(url, target_path):
    """Télécharge un fichier avec suivi de progression."""
    if os.path.exists(target_path):
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        print(f"ℹ️ Le fichier {os.path.basename(target_path)} existe déjà ({size_mb:.2f} MB). Téléchargement ignoré.")
        return
    
    print(f"⬇️ Téléchargement depuis {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get("content-length", 0))
    
    with open(target_path, "wb") as f, tqdm(
        desc=os.path.basename(target_path),
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    
    size_mb = os.path.getsize(target_path) / (1024 * 1024)
    print(f"✅ Téléchargé avec succès ({size_mb:.2f} MB) -> {target_path}")

# Téléchargement des deux fichiers
download_file(TAXI_URL, taxi_local_path)
download_file(ZONE_URL, zone_local_path)


---
## 4. Ingestion dans le Data Lake MinIO (Bucket `bronze`)

Nous stockons les fichiers dans le bucket `bronze` selon une arborescence claire et partitionnée :
- `nyc_taxi/raw_year=2024/raw_month=01/yellow_tripdata_2024-01.parquet`
- `reference/taxi_zone_lookup.csv`


In [ ]:
s3_client = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "admin"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "password123"),
    region_name="us-east-1"
)

# Clés de stockage dans MinIO Bronze
s3_taxi_key = f"nyc_taxi/raw_year={DATASET_YEAR}/raw_month={DATASET_MONTH}/yellow_tripdata_{DATASET_YEAR}-{DATASET_MONTH}.parquet"
s3_zone_key = "reference/taxi_zone_lookup.csv"

def upload_to_bronze(local_path, s3_key):
    print(f"📤 Envoi de {os.path.basename(local_path)} vers MinIO s3://bronze/{s3_key}...")
    s3_client.upload_file(local_path, "bronze", s3_key)
    print("   ✓ Envoi terminé.")

upload_to_bronze(taxi_local_path, s3_taxi_key)
upload_to_bronze(zone_local_path, s3_zone_key)


---
## 5. Inspection de la Couche Bronze via PySpark

À cette étape, nous inspectons les données brutes **sans les modifier**.
Observons le schéma brut et comptons le nombre de lignes réelles enregistrées par la ville de New York.


In [ ]:
from pyspark.sql import SparkSession

minio_endpoint = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.getenv("MINIO_ACCESS_KEY", "admin")
minio_secret_key = os.getenv("MINIO_SECRET_KEY", "password123")

spark = (
    SparkSession.builder
    .appName("01_Ingestion_Bronze")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

bronze_taxi_s3a = f"s3a://bronze/{s3_taxi_key}"
print(f"📖 Lecture directe de la couche Bronze : {bronze_taxi_s3a}")

df_bronze = spark.read.parquet(bronze_taxi_s3a)

print(f"\n📊 Nombre de colonnes : {len(df_bronze.columns)}")
print("📋 Schéma brut :")
df_bronze.printSchema()

raw_count = df_bronze.count()
print(f"\n🔢 Nombre total de courses brutes ingérées : {raw_count:,} lignes")


### Aperçu des 5 premières courses brutes


In [ ]:
df_bronze.select(
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount"
).show(5, truncate=False)


### Lecture de la table de référence des Zones


In [ ]:
bronze_zone_s3a = f"s3a://bronze/{s3_zone_key}"
df_zones_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv(bronze_zone_s3a)

print(f"📍 Nombre total de zones géographiques : {df_zones_bronze.count()}")
df_zones_bronze.show(5)


---
## 🏁 Conclusion du Notebook 01
- La couche **Bronze** est alimentée avec succès dans **MinIO**.
- Nous disposons de l'intégralité du jeu de données brut (~3 millions de courses) et du référentiel des zones.
- **Rappel** : Nous n'avons effectué aucun filtre ni nettoyage. Si nous regardons de près, certaines courses comportent des montants négatifs ou des anomalies !

👉 Rendez-vous dans le notebook **`02_transformation_silver.ipynb`** pour diagnostiquer ces anomalies et concevoir notre pipeline de nettoyage et d'enrichissement !
